# เฉลย Lab 01 — โมเดลแรกของคุณ

> คำแนะนำการใช้เฉลย: ลองพยายามทำเองก่อน แล้วค่อยเปิดเทียบทีละข้อ
> ถ้าผลลัพธ์ตัวเลขไม่เหมือนกัน ให้เช็ก `random_state` ก่อนอย่างอื่น

In [1]:
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd

X, y = load_iris(return_X_y=True)

### ข้อ 1 — แบ่ง 70:30

`test_size=0.3` คือสัดส่วนชุดทดสอบ · `random_state=42` ทำให้การแบ่งทำซ้ำได้ (ทุกคนได้ split เดียวกัน)

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)
print("train:", X_train.shape, "| test:", X_test.shape)

train: (105, 4) | test: (45, 4)


### ข้อ 2 — เทรน + คะแนน

In [3]:
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)
print("test accuracy:", model.score(X_test, y_test))

test accuracy: 1.0


ได้ 1.0 — iris ง่ายใจดี อย่าคาดหวังเลขสวยขนาดนี้กับข้อมูลจริงนะครับ (Titanic รออยู่คาบหน้า)

### ข้อ 3 — loop ทดลองค่า k

In [4]:
for k in [1, 3, 5, 15, 50]:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train, y_train)
    print(f"k={k:2d}  accuracy={m.score(X_test, y_test):.3f}")

k= 1  accuracy=1.000
k= 3  accuracy=1.000
k= 5  accuracy=1.000
k=15  accuracy=1.000
k=50  accuracy=0.956


**คำตอบชวนคุย:** k ตั้งแต่ 1-15 ได้ 1.000 หมด (ข้อมูลง่ายมาก) แต่ k=50 ตกเหลือ 0.956
เพราะเวลาโหวต เพื่อนบ้าน 50 ตัว "กว้างเกินไป" — เสียงจากดอกไม้สายพันธุ์อื่นเข้ามาปน จนครอบรูปแบบเฉพาะถิ่น (underfitting)
ในทางกลับกัน k=1 อ่อนไหวต่อ noise มาก (overfitting) — แค่ข้อมูล iris ง่ายเกินจึงมองไม่เห็น

### ข้อ 4 — แถวที่ทายผิด

ใช้ k=50 เพื่อให้มีข้อผิดพลาดให้ดู (k เล็กพลาด 0 แถว)

In [5]:
model50 = KNeighborsClassifier(n_neighbors=50).fit(X_train, y_train)
y_pred = model50.predict(X_test)

compare = pd.DataFrame({"actual": y_test, "predicted": y_pred})
compare[compare["actual"] != compare["predicted"]]

,actual,predicted
21,2,1
40,2,1


พลาด 2 แถว และพลาดทางเดียวกัน: ทาย virginica (2) เป็น versicolor (1) — สองสายพันธุ์นี้ซ้อนกันในเชิงขนาด

### ข้อ 5 — wine dataset

In [6]:
Xw, yw = load_wine(return_X_y=True)
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.3, random_state=42)

for k in [1, 3, 5, 15]:
    m = KNeighborsClassifier(n_neighbors=k).fit(Xw_tr, yw_tr)
    print(f"wine k={k:2d}  accuracy={m.score(Xw_te, yw_te):.3f}")

wine k= 1  accuracy=0.796
wine k= 3  accuracy=0.741
wine k= 5  accuracy=0.741
wine k=15  accuracy=0.759


wine ยากกว่า iris เล็กน้อย (ฟีเจอร์ 13 ตัว scale ต่างกันมาก เช่น proline ~1000 vs hue ~1)
ถ้าคะแนนแปลก ๆ — นั่นแหละคือ "ปัญหา scale" ที่คาบ 3 เราจะแก้ด้วย StandardScaler!